# Gradient descent : ajuster une droite

**Objectifs.** Calculer une loss, relier pente et dérivée, coder une mise à jour de gradient et visualiser l’effet du pas. Durée : 60–90 min.

On cherche les paramètres `w` et `b` d’une droite `y_hat = w*x + b`. Les données ci-dessous sont fabriquées selon `y = 3x + 2` avec un bruit léger.

## 1. Loss quadratique

Pour `n` points, \(J(w,b)=\frac{1}{n}\sum_i(wx_i+b-y_i)^2\). La loss est grande lorsque les prédictions s’éloignent des observations. Le but est d’en trouver un minimum, pas d’obtenir une pente de zéro.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(3)
x = np.linspace(-2, 2, 80)
y = 3 * x + 2 + rng.normal(0, 0.25, size=x.size)
plt.scatter(x, y, s=18)
plt.xlabel('x'); plt.ylabel('y')
plt.title('Exemples observés')
plt.show()

## 2. Dérivées partielles

Avec `error = prediction - y`, on obtient \(\partial J/\partial w = 2\,mean(error\cdot x)\) et \(\partial J/\partial b = 2\,mean(error)\). Les formes NumPy sont des scalaires : les deux paramètres sont partagés par toutes les observations.

In [ ]:
def mse(weight, bias, features, targets):
    predictions = weight * features + bias
    return np.mean((predictions - targets) ** 2)

weight, bias = 0.0, 0.0
error = weight * x + bias - y
grad_weight = 2 * np.mean(error * x)
grad_bias = 2 * np.mean(error)
print('J(0, 0) =', round(float(mse(weight, bias, x, y)), 3))
print('gradient (w, b) =', round(float(grad_weight), 3), round(float(grad_bias), 3))

### Un pas calculé à la main

Avec `x=[0,1]`, `y=[1,3]` et `w=b=0`, les prédictions sont `[0,0]` et les erreurs `[-1,-3]`. La MSE est `(1²+3²)/2=5`. Les dérivées valent `dJ/dw = 2·mean(error*x) = -3` et `dJ/db = 2·mean(error) = -4`. Un pas de taille `0.1` donne donc `w=0.3`, `b=0.4`. Vérifions que la loss baisse.

In [ ]:
small_x, small_y = np.array([0., 1.]), np.array([1., 3.])
small_w, small_b, step = 0., 0., 0.1
small_error = small_w * small_x + small_b - small_y
small_grad_w = 2 * np.mean(small_error * small_x)
small_grad_b = 2 * np.mean(small_error)
next_w = small_w - step * small_grad_w
next_b = small_b - step * small_grad_b
print('gradients :', small_grad_w, small_grad_b, '| loss initiale :', mse(small_w, small_b, small_x, small_y), '| loss après un pas :', mse(next_w, next_b, small_x, small_y))

## 3. Implémenter une mise à jour

Soustraire une fraction du gradient déplace les paramètres vers une région de loss plus faible. `learning_rate` est cette fraction. On enregistre la loss avant chaque mise à jour pour examiner l’apprentissage.

In [ ]:
learning_rate = 0.08
epochs = 300
weight, bias = 0.0, 0.0
loss_history = []
for _ in range(epochs):
    prediction = weight * x + bias
    error = prediction - y
    loss_history.append(float(np.mean(error ** 2)))
    grad_weight = 2 * np.mean(error * x)
    grad_bias = 2 * np.mean(error)
    weight -= learning_rate * grad_weight
    bias -= learning_rate * grad_bias
print(f'w={weight:.3f}, b={bias:.3f}, final MSE={loss_history[-1]:.4f}')

## 4. Visualiser la convergence

Une courbe descendante indique que cet entraînement réduit la loss sur ces points. Cela ne garantit pas une bonne généralisation, ni que toute learning rate fonctionnera.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(loss_history)
axes[0].set(xlabel='Epoch', ylabel='MSE', title='Loss pendant l’entraînement')
axes[1].scatter(x, y, s=18, label='Données')
axes[1].plot(x, weight * x + bias, color='darkorange', label='Droite ajustée')
axes[1].legend(); axes[1].set(xlabel='x', ylabel='y', title='Prédiction finale')
plt.tight_layout(); plt.show()

### Visualiser le trajet dans l'espace des paramètres

La courbe de loss montre *quand* le modèle progresse. Les courbes de niveau montrent *où* il se déplace : chaque point est une paire `(w,b)`, et les régions foncées ont une MSE plus faible. Les axes sont des paramètres, pas des observations. La trajectoire suivante reprend la même learning rate et les mêmes données sur 35 pas.

In [ ]:
grid_w, grid_b = np.meshgrid(np.linspace(0, 4.5, 90), np.linspace(0, 3.2, 90))
loss_surface = np.mean((grid_w[..., None] * x + grid_b[..., None] - y) ** 2, axis=-1)
trace = [(0., 0.)]
trace_w, trace_b = 0., 0.
for _ in range(35):
    trace_error = trace_w * x + trace_b - y
    trace_w -= learning_rate * 2 * np.mean(trace_error * x)
    trace_b -= learning_rate * 2 * np.mean(trace_error)
    trace.append((trace_w, trace_b))
trace = np.asarray(trace)
fig, ax = plt.subplots(figsize=(6, 5))
levels = ax.contourf(grid_w, grid_b, loss_surface, levels=25, cmap='viridis')
ax.plot(trace[:, 0], trace[:, 1], 'o-', color='white', markersize=3, linewidth=1, label='Pas successifs')
ax.set(xlabel='Pente w', ylabel='Biais b', title='Trajet de gradient descent sur la MSE')
fig.colorbar(levels, ax=ax, label='MSE'); ax.legend(); plt.show()

**Interprétation :** la trajectoire se rapproche d'un minimum, mais le point exact dépend des données, du pas et du nombre d'epochs. **Contrôle numérique :** pour vérifier une dérivée, calculer `[J(w+h,b)-J(w-h,b)]/(2h)` avec un petit `h`, puis comparer à la formule analytique.

## 5. Expérimenter avec le pas

Un pas trop grand peut osciller ou diverger. Cette cellule mesure la loss après 40 mises à jour pour plusieurs taux. Si la valeur devient non finie, la descente a divergé.

In [ ]:
for rate in [0.01, 0.2, 0.8, 1.5]:
    w, b = 0.0, 0.0
    for _ in range(40):
        err = w * x + b - y
        w -= rate * 2 * np.mean(err * x)
        b -= rate * 2 * np.mean(err)
    print(f'learning_rate={rate:>4}: MSE={mse(w, b, x, y):.3g}')

## 6. Interpréter et relier au code

`x` et `y` ont la forme `(80,)`. `prediction` et `error` gardent cette forme. Le gradient agrège les 80 contributions par `mean`, ce qui produit un nombre pour chaque paramètre. Les paramètres approchent 3 et 2, mais le bruit et la longueur finie du calcul empêchent une égalité parfaite.

**Limites :** MSE pénalise les grosses erreurs au carré ; les variables à des échelles très différentes peuvent rendre le choix du pas difficile ; les minima locaux et dimensions multiples rendent les problèmes réels plus complexes.

## À retenir et exercices

Gradient descent est une méthode numérique générale, pas une garantie automatique. Comparez avec la solution analytique simple, dessinez une mise à jour sur papier, puis modifiez la graine. Exercices : [`02_gradients.ipynb`](../../exercises/intermediate/02_gradients.ipynb).